# LC 39 — Combination Sum
**Day 39 | Theme: Backtracking | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">

**Core Insight:** Unlike Subsets, you may reuse the same element
unlimited times — so recurse with `i` (not `i+1`). Sorting lets
you prune the entire remaining branch the instant a candidate
exceeds the remaining target.

</div>

## Official Problem Statement

Given an array of **distinct** integers `candidates` and a
target integer `target`, return *a list of all **unique**
combinations of `candidates` where the chosen numbers sum
to `target`*.

You may return the combinations in **any order**.
The **same** number may be chosen from `candidates` an
**unlimited number of times**.
Two combinations are unique if the frequency of at least
one of the chosen numbers is different.

**Constraints:**
- `1 <= candidates.length <= 30`
- `2 <= candidates[i] <= 40`
- All elements of `candidates` are **distinct**.
- `1 <= target <= 500`

## What This Is Actually Asking

Find every multiset of numbers from `candidates` that adds
up to exactly `target`, where each candidate can appear
more than once.
Order inside a combination does not matter: `[2,2,3]` and
`[3,2,2]` are the same answer.
You need all such combinations, not just one.
The reuse-allowed rule is what separates this from LC 40
(Combination Sum II, where each element is used once).

## Walk Through an Example by Hand

`candidates = [2, 3, 6, 7]`, `target = 7`

```
Sort: [2, 3, 6, 7]

backtrack(start=0, path=[], remain=7)
  i=0 cand=2, remain=7-2=5 -> backtrack(0, [2], 5)
    i=0 cand=2, remain=5-2=3 -> backtrack(0, [2,2], 3)
      i=0 cand=2, remain=3-2=1 -> backtrack(0,[2,2,2],1)
        i=0 cand=2 > remain=1: BREAK
      i=1 cand=3 > remain=1: BREAK (after pop)
      pop -> [2,2]
      i=1 cand=3, remain=3-3=0 -> FOUND [2,2,3]
      i=2 cand=6 > remain=3: BREAK
    pop -> [2]
    i=1 cand=3, remain=5-3=2 -> backtrack(1,[2,3],2)
      i=1 cand=3 > remain=2: BREAK
    pop -> [2]
    i=2 cand=6 > remain=5: BREAK
  pop -> []
  i=1 cand=3, remain=7-3=4 -> backtrack(1,[3],4)
    i=1 cand=3, remain=4-3=1 -> backtrack(1,[3,3],1)
      i=1 cand=3 > 1: BREAK
    pop->[3]  i=2 cand=6>4: BREAK
  pop -> []
  i=2 cand=6, remain=7-6=1 -> backtrack(2,[6],1)
      i=2 cand=6 > 1: BREAK
  pop -> []
  i=3 cand=7, remain=7-7=0 -> FOUND [7]

Result: [[2,2,3], [7]]
```

## The Picture

```
target=7, candidates=[2,3,6,7]

                  remain=7
          /        |      |      \
        -2        -3     -6      -7
       [2]       [3]    [6]     [7]*
       / |\       |\     |
     -2 -3 -6   -3 -6  -6>1
    [2,2][2,3] [3,3][3,6]  PRUNE
     /\    |
   -2 -3* -3
  [2,2,2][2,2,3]* [2,3,3]
   PRUNE          1<3 PRUNE

* = valid combination (remain==0)
PRUNE = cand > remain (sorted order lets us break)
```

**Key:** recurse with `i` (same index) to allow reuse.
Use `break` (not `continue`) once `cand > remain`
because all later candidates are even larger (sorted).

## When To Use This Pattern

- When the problem says "combinations that sum to target",
  think backtracking with a `remain` counter.
- When elements may be **reused**, think recurse with `i`
  instead of `i+1`.
- When elements **cannot** be reused, think `i+1` (LC 40).
- When the input is sorted and you can prune, think
  `if cand > remain: break` for early exit.
- When you need count (not list) of combinations, think
  dynamic programming instead of backtracking.

## The Approach

Sort `candidates` so that pruning works: once a candidate
exceeds `remain`, all subsequent candidates will too.
In `backtrack(start, path, remain)`, loop from `start`;
if `cand == remain` append `path + [cand]` and return;
if `cand < remain` append it, recurse with `i` (reuse
allowed), then pop.
Break as soon as `cand > remain` to prune the subtree.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    """Order-independent combination equality check."""
    def norm(result):
        return sorted(tuple(sorted(s)) for s in result)

    cases = [
        # (candidates, target, expected)
        ([2, 3, 6, 7], 7,
         [[2, 2, 3], [7]]),
        ([2, 3, 5], 8,
         [[2, 2, 2, 2], [2, 3, 3], [3, 5]]),
        ([2], 1,
         []),                          # no combination possible
        ([1], 1,
         [[1]]),
        ([1], 2,
         [[1, 1]]),
    ]

    passed = 0
    for i, (candidates, target, expected) in enumerate(cases):
        res = func(candidates, target)
        if norm(res) == norm(expected):
            print(f"  Case {i+1}: PASSED")
            passed += 1
        else:
            print(f"  Case {i+1}: FAILED")
            print(f"    Input    : candidates={candidates},"
                  f" target={target}")
            print(f"    Expected : {norm(expected)}")
            print(f"    Got      : {norm(res)}")

    print(f"\n  Summary: {passed}/{len(cases)} passed")

In [5]:
def combinationSum(
    candidates: List[int], target: int
) -> List[List[int]]:
    
    result = []

    def backtrack (start, current, remaining):
        if remaining == 0 :
            result.append(current[:])
            return
   
        if remaining < 0:
            return

        for i in range ( start , len(candidates)):
            current.append(candidates[i])
            backtrack(i , current, remaining - candidates[i])
            current.pop()
    backtrack (0, [] , target)
    return result




    
    



#Quick debug — run this cell while building
print(combinationSum([2,3,6,7], 7))   # [[2,2,3],[7]]
print(combinationSum([2,3,5], 8))     # [[2,2,2,2],[2,3,3],[3,5]]
print(combinationSum([2], 1))          # []
print(combinationSum([1], 2))          # [[1,1]]
test_harness(combinationSum)    


[[2, 2, 3], [7]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[]
[[1, 1]]
  Case 1: PASSED
  Case 2: PASSED
  Case 3: PASSED
  Case 4: PASSED
  Case 5: PASSED

  Summary: 5/5 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(combinationSum)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (no prune) | O(n^(T/M)) | O(T/M) | All paths explored |
| Backtrack + sort+prune | O(n^(T/M)) | O(T/M) | Prune cuts many branches |
| DP (count only) | O(n·T) | O(T) | Only if you need count |

- `T` = target, `M` = minimum candidate value.
- Worst case depth = `T/M` (all smallest elements).
- Sorting + `break` prunes branches where the rest of the
  loop would also fail — significant constant-factor win.
- DP is asymptotically better but only produces a count,
  not the actual combinations.

## Real World Connection

At **Citi**, portfolio construction sometimes requires finding
all allocations of capital across a set of instruments that
hit an exact notional target — a direct analogue of combination
sum with reuse.
In **AWS cost optimisation**, selecting Reserved Instance
configurations (1-year, 3-year, different sizes) to cover a
compute budget exactly mirrors this pattern.
**Data pipeline scheduling** can use combination-sum logic
to find sets of job run-times that fill a fixed maintenance
window without overflow.
The pruning strategy (sort + break) is a microcosm of
branch-and-bound, the backbone of industrial integer
programming solvers used in logistics and finance.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra